# SHL Hiring Assessment 2026
## Spoken Grammar Scoring Engine

### Objective

The objective is to predict a continuous grammar score between 0 and 5 from spoken audio samples.

Each audio file contains approximately 45–60 seconds of speech.

### Approach

The solution uses three types of information:

1. Speech transcription
   - Convert audio into text using Whisper.
   - Grammar is mainly reflected in the language used by the speaker.

2. Text-based features
   - Word and character TF-IDF features.
   - Sentence length.
   - Vocabulary usage.
   - Repeated words.
   - Filler words.
   - Basic sentence structure indicators.

3. Audio-based features
   - Duration.
   - RMS energy.
   - Zero crossing rate.
   - Spectral features.
   - Pitch-related statistics.

The text and audio features are combined and used with a regression model.

### Evaluation

The model is evaluated using:

- RMSE
- Pearson Correlation

A 5-fold cross-validation setup is used to estimate model performance on unseen training samples.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 1. Import libraries

In [ ]:
import os
import re
import warnings
import subprocess
import sys

import numpy as np
import pandas as pd

import librosa
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.pipeline import FeatureUnion

from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# 2. Find the dataset

In [ ]:
BASE_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")

TRAIN_AUDIO = os.path.join(BASE_PATH, "train")
TEST_AUDIO = os.path.join(BASE_PATH, "test")

train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

display(train.head())
display(test.head())

# 3. Automatically identify columns
This makes the notebook safer if the column names differ slightly.



In [ ]:
def find_column(df, possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    return None


file_col = find_column(
    train,
    ["filename", "file_name", "audio", "audio_file", "id"]
)

target_col = find_column(
    train,
    ["label", "score", "grammar_score", "target", "MOS"]
)

print("File column:", file_col)
print("Target column:", target_col)

if file_col is None:
    raise ValueError("Could not identify the audio filename column.")

if target_col is None:
    raise ValueError("Could not identify the target/grammar score column.")

# 4. Basic dataset check

In [ ]:
print("Number of training samples:", len(train))
print("Number of test samples:", len(test))

print("\nMissing values:")
print(train.isnull().sum())

print("\nTarget statistics:")
print(train[target_col].describe())

print("\nScore distribution:")
display(train[target_col].value_counts().sort_index())

# 5. Check audio files

In [ ]:
train["audio_path"] = train[file_col].apply(
    lambda x: os.path.join(TRAIN_AUDIO, str(x))
)

test["audio_path"] = test[file_col].apply(
    lambda x: os.path.join(TEST_AUDIO, str(x))
)

print("Missing train audio files:",
      (~train["audio_path"].apply(os.path.exists)).sum())

print("Missing test audio files:",
      (~test["audio_path"].apply(os.path.exists)).sum())

# 6. Check one audio file

In [ ]:
sample_file = train.iloc[0]["audio_path"]

y, sr = librosa.load(sample_file, sr=None)

print("Sample rate:", sr)
print("Duration:", round(len(y) / sr, 2), "seconds")
print("Number of samples:", len(y))

# 7. Target distribution

In [ ]:
# Target distribution

plt.figure(figsize=(8, 5))

plt.hist(
    train[target_col],
    bins=np.arange(0.5, 5.6, 0.5),
    edgecolor="black"
)

plt.xlabel("Grammar Score")
plt.ylabel("Number of Samples")
plt.title("Distribution of Grammar Scores")
plt.xticks([1, 2, 3, 4, 5])
plt.grid(alpha=0.2)

plt.show()

In [ ]:
# Exact number of samples for each score

score_distribution = (
    train[target_col]
    .value_counts()
    .sort_index()
    .rename_axis("Grammar Score")
    .reset_index(name="Number of Samples")
)

display(score_distribution)

In [ ]:
# Percentage distribution

score_distribution["Percentage"] = (
    score_distribution["Number of Samples"]
    / len(train)
    * 100
)

score_distribution["Percentage"] = (
    score_distribution["Percentage"]
    .round(2)
)

display(score_distribution)

# 8. Check Audio Quality
Before extracting features, let's inspect the recordings.

In [ ]:
def inspect_audio(audio_path):

    y, sr = librosa.load(
        audio_path,
        sr=None,
        mono=True
    )

    duration = len(y) / sr

    return {
        "sample_rate": sr,
        "duration": duration,
        "samples": len(y),
        "mean_amplitude": np.mean(np.abs(y)),
        "max_amplitude": np.max(np.abs(y))
    }


sample_audio_info = inspect_audio(
    train.iloc[0]["audio_path"]
)

sample_audio_info

# 9. Analyze All Training Audio

In [ ]:
audio_summary = []

for path in tqdm(
    train["audio_path"],
    desc="Checking training audio"
):

    try:

        y, sr = librosa.load(
            path,
            sr=None,
            mono=True
        )

        audio_summary.append({
            "duration": len(y) / sr,
            "sample_rate": sr,
            "mean_amplitude": np.mean(np.abs(y)),
            "max_amplitude": np.max(np.abs(y))
        })

    except Exception:

        audio_summary.append({
            "duration": np.nan,
            "sample_rate": np.nan,
            "mean_amplitude": np.nan,
            "max_amplitude": np.nan
        })


audio_summary = pd.DataFrame(audio_summary)

display(audio_summary.describe())

# 10. Duration Distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    audio_summary["duration"].dropna(),
    bins=20,
    edgecolor="black"
)

plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Audio Files")
plt.title("Audio Duration Distribution")
plt.grid(alpha=0.2)

plt.show()

# 11. Check for Broken Audio

In [ ]:
print(
    "Missing duration:",
    audio_summary["duration"].isna().sum()
)

print(
    "Very short files:",
    (audio_summary["duration"] < 20).sum()
)

print(
    "Very long files:",
    (audio_summary["duration"] > 90).sum()
)

# 12. Audio Feature Extraction
Since we cannot install another speech-recognition package, we'll make the audio model the foundation.

For grammar scoring, useful speech characteristics include:

- speaking duration
- energy
- pauses/silence
- zero crossing rate
- spectral characteristics
- MFCC
- pitch
- speech activity ratio

We can extract these directly from the WAV files.

# 13. Create a Strong Audio Feature Function

In [ ]:
def extract_audio_features(audio_path):

    try:

        y, sr = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )

        # Remove very small background noise
        y_trimmed, _ = librosa.effects.trim(
            y,
            top_db=30
        )

        duration = len(y) / sr
        speech_duration = len(y_trimmed) / sr

        # -------------------------
        # Energy
        # -------------------------

        rms = librosa.feature.rms(
            y=y
        )[0]

        rms_mean = np.mean(rms)
        rms_std = np.std(rms)

        rms_min = np.min(rms)
        rms_max = np.max(rms)

        # -------------------------
        # Zero crossing rate
        # -------------------------

        zcr = librosa.feature.zero_crossing_rate(
            y
        )[0]

        zcr_mean = np.mean(zcr)
        zcr_std = np.std(zcr)

        # -------------------------
        # Spectral features
        # -------------------------

        spectral_centroid = librosa.feature.spectral_centroid(
            y=y,
            sr=sr
        )[0]

        spectral_bandwidth = librosa.feature.spectral_bandwidth(
            y=y,
            sr=sr
        )[0]

        spectral_rolloff = librosa.feature.spectral_rolloff(
            y=y,
            sr=sr
        )[0]

        # -------------------------
        # MFCC
        # -------------------------

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=13
        )

        # -------------------------
        # Pitch
        # -------------------------

        f0, voiced_flag, voiced_prob = librosa.pyin(
            y,
            fmin=70,
            fmax=400,
            sr=sr
        )

        valid_pitch = f0[
            ~np.isnan(f0)
        ]

        if len(valid_pitch) > 0:

            pitch_mean = np.mean(valid_pitch)
            pitch_std = np.std(valid_pitch)
            pitch_min = np.min(valid_pitch)
            pitch_max = np.max(valid_pitch)

        else:

            pitch_mean = 0
            pitch_std = 0
            pitch_min = 0
            pitch_max = 0

        # -------------------------
        # Voice activity
        # -------------------------

        if len(y) > 0:

            speech_ratio = (
                speech_duration / duration
            )

        else:

            speech_ratio = 0

        features = {

            "duration": duration,

            "speech_duration": speech_duration,

            "speech_ratio": speech_ratio,

            "rms_mean": rms_mean,
            "rms_std": rms_std,
            "rms_min": rms_min,
            "rms_max": rms_max,

            "zcr_mean": zcr_mean,
            "zcr_std": zcr_std,

            "spectral_centroid_mean":
                np.mean(spectral_centroid),

            "spectral_centroid_std":
                np.std(spectral_centroid),

            "spectral_bandwidth_mean":
                np.mean(spectral_bandwidth),

            "spectral_bandwidth_std":
                np.std(spectral_bandwidth),

            "spectral_rolloff_mean":
                np.mean(spectral_rolloff),

            "spectral_rolloff_std":
                np.std(spectral_rolloff),

            "pitch_mean": pitch_mean,
            "pitch_std": pitch_std,
            "pitch_min": pitch_min,
            "pitch_max": pitch_max
        }

        # Add MFCC statistics

        for i in range(13):

            features[
                f"mfcc_{i+1}_mean"
            ] = np.mean(mfcc[i])

            features[
                f"mfcc_{i+1}_std"
            ] = np.std(mfcc[i])

        return features

    except Exception as e:

        print(
            "Error processing:",
            audio_path,
            e
        )

        return {}

# 14. Extract Training Features

In [ ]:
train_feature_cache = (
    "/kaggle/working/train_audio_features.csv"
)

if os.path.exists(train_feature_cache):

    train_audio_features = pd.read_csv(
        train_feature_cache
    )

    print("Loaded cached features.")

else:

    feature_list = []

    for path in tqdm(
        train["audio_path"],
        desc="Extracting training audio features"
    ):

        feature_list.append(
            extract_audio_features(path)
        )

    train_audio_features = pd.DataFrame(
        feature_list
    )

    train_audio_features.to_csv(
        train_feature_cache,
        index=False
    )

print(
    "Training feature shape:",
    train_audio_features.shape
)

display(
    train_audio_features.head()
)

# 15. Extract Test Features

In [ ]:
test_feature_cache = (
    "/kaggle/working/test_audio_features.csv"
)

if os.path.exists(test_feature_cache):

    test_audio_features = pd.read_csv(
        test_feature_cache
    )

    print("Loaded cached features.")

else:

    feature_list = []

    for path in tqdm(
        test["audio_path"],
        desc="Extracting test audio features"
    ):

        feature_list.append(
            extract_audio_features(path)
        )

    test_audio_features = pd.DataFrame(
        feature_list
    )

    test_audio_features.to_csv(
        test_feature_cache,
        index=False
    )

print(
    "Test feature shape:",
    test_audio_features.shape
)

# 16. Clean Feature Data

In [ ]:
train_audio_features = (
    train_audio_features
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

test_audio_features = (
    test_audio_features
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    "Missing training values:",
    train_audio_features.isna().sum().sum()
)

print(
    "Missing test values:",
    test_audio_features.isna().sum().sum()
)

# 17. Correlation Between Features and Grammar

This is a useful visualization for your report.

In [ ]:
feature_correlation = (
    train_audio_features
    .corrwith(train[target_col])
    .sort_values()
)

display(
    feature_correlation.to_frame(
        "Correlation"
    )
)

# 18. Plot Most Relevant Features

In [ ]:
top_features = pd.concat([
    feature_correlation.head(8),
    feature_correlation.tail(8)
])

plt.figure(figsize=(10, 7))

top_features.sort_values().plot(
    kind="barh"
)

plt.xlabel("Pearson Correlation")
plt.ylabel("Audio Feature")
plt.title(
    "Audio Features Most Related to Grammar Score"
)

plt.grid(alpha=0.2)

plt.show()

# 19. Build the Main Training Matrix

At this stage, don't use the transcript-related cells from my previous answer because faster-whisper could not be installed.

We'll build the model using robust audio features first.

In [ ]:
X = train_audio_features.copy()
X_test = test_audio_features.copy()

y = train[target_col].values

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("y shape:", y.shape)

# 20. Standardize Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_test_scaled = scaler.transform(
    X_test
)

print(
    "Scaled training shape:",
    X_scaled.shape
)

# 21. Baseline Model

Before using a machine-learning model, calculate a simple baseline.

The baseline predicts the average training score for every sample.

In [ ]:
baseline_prediction = np.full(
    len(y),
    np.mean(y)
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y,
        baseline_prediction
    )
)

print(
    "Baseline RMSE:",
    round(baseline_rmse, 4)
)

# 22. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_oof = np.zeros(len(y))

rf_results = []

for fold, (train_idx, valid_idx) in enumerate(
    kf.split(X_scaled),
    start=1
):

    X_train = X_scaled[train_idx]
    X_valid = X_scaled[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_valid
    )

    pred = np.clip(
        pred,
        0,
        5
    )

    rf_oof[valid_idx] = pred

    rmse = np.sqrt(
        mean_squared_error(
            y_valid,
            pred
        )
    )

    pearson = pearsonr(
        y_valid,
        pred
    )[0]

    rf_results.append({
        "Fold": fold,
        "RMSE": rmse,
        "Pearson": pearson
    })

    print(
        f"Fold {fold}: "
        f"RMSE={rmse:.4f}, "
        f"Pearson={pearson:.4f}"
    )

# 23. Random Forest Overall Result

In [ ]:
rf_rmse = np.sqrt(
    mean_squared_error(
        y,
        rf_oof
    )
)

rf_pearson = pearsonr(
    y,
    rf_oof
)[0]

print("=" * 50)
print("RANDOM FOREST OOF RESULT")
print("=" * 50)

print(
    f"Training OOF RMSE: {rf_rmse:.4f}"
)

print(
    f"Training OOF Pearson: {rf_pearson:.4f}"
)

# 24. Extra Trees

This is worth testing because ExtraTrees often performs nicely on small tabular datasets.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et_oof = np.zeros(len(y))

et_results = []

for fold, (train_idx, valid_idx) in enumerate(
    kf.split(X_scaled),
    start=1
):

    model = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_scaled[train_idx],
        y[train_idx]
    )

    pred = model.predict(
        X_scaled[valid_idx]
    )

    pred = np.clip(
        pred,
        0,
        5
    )

    et_oof[valid_idx] = pred

    rmse = np.sqrt(
        mean_squared_error(
            y[valid_idx],
            pred
        )
    )

    pearson = pearsonr(
        y[valid_idx],
        pred
    )[0]

    et_results.append({
        "Fold": fold,
        "RMSE": rmse,
        "Pearson": pearson
    })

    print(
        f"Fold {fold}: "
        f"RMSE={rmse:.4f}, "
        f"Pearson={pearson:.4f}"
    )

# 25. Extra Trees Overall Result

In [ ]:
et_rmse = np.sqrt(
    mean_squared_error(
        y,
        et_oof
    )
)

et_pearson = pearsonr(
    y,
    et_oof
)[0]

print("=" * 50)
print("EXTRA TREES OOF RESULT")
print("=" * 50)

print(
    f"Training OOF RMSE: {et_rmse:.4f}"
)

print(
    f"Training OOF Pearson: {et_pearson:.4f}"
)

# 26. Compare Models

In [ ]:
model_results = pd.DataFrame({
    "Model": [
        "Mean Baseline",
        "Random Forest",
        "Extra Trees"
    ],
    "RMSE": [
        baseline_rmse,
        rf_rmse,
        et_rmse
    ],
    "Pearson": [
        pearsonr(y, baseline_prediction)[0],
        rf_pearson,
        et_pearson
    ]
})

display(
    model_results.sort_values(
        "RMSE"
    )
)

# 27. Actual vs Predicted

Use the better model based on CV.

In [ ]:
if et_rmse < rf_rmse:

    best_oof = et_oof
    best_model_name = "Extra Trees"

else:

    best_oof = rf_oof
    best_model_name = "Random Forest"


best_rmse = np.sqrt(
    mean_squared_error(
        y,
        best_oof
    )
)

best_pearson = pearsonr(
    y,
    best_oof
)[0]

print("Selected model:", best_model_name)
print("OOF RMSE:", round(best_rmse, 4))
print("OOF Pearson:", round(best_pearson, 4))

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y,
    best_oof,
    alpha=0.65
)

plt.plot(
    [0, 5],
    [0, 5],
    linestyle="--"
)

plt.xlabel("Actual Grammar Score")
plt.ylabel("Predicted Grammar Score")

plt.title(
    f"Actual vs Predicted - {best_model_name}"
)

plt.xlim(0, 5)
plt.ylim(0, 5)

plt.grid(alpha=0.2)

plt.show()

# 28. Residual Analysis

In [ ]:
residuals = y - best_oof

plt.figure(figsize=(8, 5))

plt.scatter(
    best_oof,
    residuals,
    alpha=0.6
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Predicted Grammar Score")
plt.ylabel("Residual")

plt.title("Residual Analysis")

plt.grid(alpha=0.2)

plt.show()

# 29. Error Distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    residuals,
    bins=25,
    edgecolor="black"
)

plt.xlabel("Prediction Error")
plt.ylabel("Number of Samples")

plt.title("Distribution of Prediction Errors")

plt.grid(alpha=0.2)

plt.show()

# 30. Feature Importance

This is another good interpretability section.

In [ ]:
if best_model_name == "Extra Trees":

    final_temp_model = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    )

else:

    final_temp_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )


final_temp_model.fit(
    X_scaled,
    y
)

importance = pd.Series(
    final_temp_model.feature_importances_,
    index=X.columns
).sort_values(
    ascending=False
)

display(
    importance.head(20)
)

In [ ]:
plt.figure(figsize=(10, 7))

importance.head(15).sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Audio Features")

plt.grid(alpha=0.2)

plt.show()

# 31. Train Final Model

Once you've compared the models, train the selected model using all 769 training samples.

In [ ]:
if best_model_name == "Extra Trees":

    final_model = ExtraTreesRegressor(
        n_estimators=700,
        max_depth=10,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    )

else:

    final_model = RandomForestRegressor(
        n_estimators=700,
        max_depth=8,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )


final_model.fit(
    X_scaled,
    y
)

print(
    "Final model trained:",
    best_model_name
)

# 32. Predict Test Set

In [ ]:
test_predictions = final_model.predict(
    X_test_scaled
)

# Keep predictions inside the valid range
test_predictions = np.clip(
    test_predictions,
    0,
    5
)

print(
    "Minimum prediction:",
    round(test_predictions.min(), 4)
)

print(
    "Maximum prediction:",
    round(test_predictions.max(), 4)
)

print(
    "Average prediction:",
    round(test_predictions.mean(), 4)
)

# 33. Check Test Predictions

In [ ]:
test_prediction_check = pd.DataFrame({
    "filename": test[file_col],
    "predicted_score": test_predictions
})

display(
    test_prediction_check.head(20)
)

# 34. Plot Test Prediction Distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    test_predictions,
    bins=20,
    edgecolor="black"
)

plt.xlabel("Predicted Grammar Score")
plt.ylabel("Number of Samples")

plt.title("Test Prediction Distribution")

plt.grid(alpha=0.2)

plt.show()

# 35. Create Submission

First inspect the exact required format.

In [ ]:
sample_submission_path = os.path.join(
    BASE_PATH,
    "sample_submission.csv"
)

sample_submission = pd.read_csv(
    sample_submission_path
)

print(
    "Sample submission columns:",
    sample_submission.columns.tolist()
)

print(
    "Sample submission shape:",
    sample_submission.shape
)

display(
    sample_submission.head()
)

# 36. Fill Sample Submission Safely

In [ ]:
submission = sample_submission.copy()

print("Submission before filling:")
display(submission.head())

In [ ]:
submission = sample_submission.copy()

print("Submission before filling:")
display(submission.head())

# 37. Final Submission Validation

In [ ]:
print("=" * 50)
print("FINAL SUBMISSION CHECK")
print("=" * 50)

print("Rows:", len(submission))

print("Expected rows:", len(test))

print(
    "Missing values:",
    submission.isna().sum().sum()
)

print(
    "Minimum prediction:",
    submission[target_col].min()
)

print(
    "Maximum prediction:",
    submission[target_col].max()
)

print(
    "Duplicate filenames:",
    submission[file_col].duplicated().sum()
)

display(
    submission.head(10)
)

# 38. Save Final Submission

In [ ]:
submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print(
    "Submission file created:"
)

print(
    submission_path
)

# 39. Final Training RMSE

This cell is mandatory for your assignment.

In [ ]:
# Check which trained models are available

for name in ["best_model", "rf_model", "et_model"]:
    print(name, "exists:", name in globals())

In [ ]:
# 39. Final Training RMSE
# Mandatory for the assignment

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Create the final model
final_model = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

# Train on all available training data
final_model.fit(X, y)

# Predict on the training data
train_predictions = final_model.predict(X)

# Keep scores within the valid 0-5 range
train_predictions = np.clip(train_predictions, 0, 5)

# Calculate mandatory training RMSE
final_training_rmse = np.sqrt(
    mean_squared_error(y, train_predictions)
)

print("Final Model: Extra Trees")
print("Training samples:", len(X))
print("Final Training RMSE:", round(final_training_rmse, 4))

In [ ]:
# Final test predictions

test_predictions = final_model.predict(X_test)

# Keep predictions within the valid score range
test_predictions = np.clip(test_predictions, 0, 5)

print("Number of test predictions:", len(test_predictions))
print("Minimum prediction:", round(test_predictions.min(), 4))
print("Maximum prediction:", round(test_predictions.max(), 4))

In [ ]:
# Create submission using filename matching

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv"
)

# Create a dataframe containing every test prediction
prediction_df = pd.DataFrame({
    "filename": test["filename"].values,
    "prediction": test_predictions
})

print("Test predictions:", len(prediction_df))
print("Sample submission rows:", len(sample_submission))

# Match predictions to the filenames in sample_submission
submission = sample_submission.copy()

submission = submission.drop(columns=["label"], errors="ignore")

submission = submission.merge(
    prediction_df,
    on="filename",
    how="left"
)

# Rename prediction column to the required label column
submission = submission.rename(
    columns={"prediction": "label"}
)

print("\nSubmission shape:", submission.shape)
display(submission.head(10))

In [ ]:
# Check whether every submission filename has a prediction

missing_predictions = submission["label"].isna().sum()

print("Missing predictions:", missing_predictions)

if missing_predictions > 0:
    print("\nFiles without predictions:")
    display(
        submission[submission["label"].isna()]
    )
else:
    print("All submission files have predictions.")

In [ ]:
# Keep predictions within valid range
submission["label"] = np.clip(
    submission["label"],
    0,
    5
)

print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

print("\nMissing values:")
print(submission.isna().sum())

print("\nPrediction range:")
print(
    "Min:", round(submission["label"].min(), 4),
    "| Max:", round(submission["label"].max(), 4)
)

In [ ]:
# Save final submission

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Final submission saved successfully:")
print(submission_path)

display(submission.head(10))

In [ ]:
# Check filename matching

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv"
)

print("Sample submission rows:", len(sample_submission))
print("Test rows:", len(test))

# Exact filename matches
matched = sample_submission["filename"].isin(test["filename"])

print("Exact filename matches:", matched.sum())
print("Not matched:", (~matched).sum())

print("\nExamples that do not match:")
display(sample_submission.loc[~matched].head(20))

In [ ]:
# Inspect test filenames and audio paths

display(
    test[["filename", "audio_path"]].head(20)
)

In [ ]:
# Compare sample submission filenames with test filenames

print("Sample submission filenames:")
print(sample_submission["filename"].head(20).tolist())

print("\nTest filenames:")
print(test["filename"].head(20).tolist())

In [ ]:
print("Missing labels:", submission["label"].isna().sum())

In [ ]:
# Diagnose the filename mismatch

print("TEST")
print("Rows:", len(test))
print("Unique filenames:", test["filename"].nunique())
print("First 20:")
print(test["filename"].head(20).tolist())

print("\n" + "="*60)

print("SAMPLE SUBMISSION")
print("Rows:", len(sample_submission))
print("Unique filenames:", sample_submission["filename"].nunique())
print("First 20:")
print(sample_submission["filename"].head(20).tolist())

print("\n" + "="*60)

# Show filenames that exist in submission but not in test
test_names = set(test["filename"].astype(str))
submission_names = set(sample_submission["filename"].astype(str))

missing_from_test = sorted(submission_names - test_names)

print("Submission filenames not found in test:", len(missing_from_test))
print(missing_from_test[:30])

print("\n" + "="*60)

# Show filenames that exist in test but not in submission
extra_in_test = sorted(test_names - submission_names)

print("Test filenames not found in submission:", len(extra_in_test))
print(extra_in_test[:30])

In [ ]:
# Inspect the actual audio paths

display(
    test[["filename", "audio_path"]].head(20)
)

print("\nSample submission:")
display(sample_submission.head(20))

In [ ]:
import os
import glob
import pandas as pd

# Load sample submission
sample_submission = pd.read_csv(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv"
)

# Find all WAV files in the competition dataset
dataset_root = "/kaggle/input/competitions/shl-hiring-assessment-2026"

all_wav_files = glob.glob(
    os.path.join(dataset_root, "**", "*.wav"),
    recursive=True
)

print("Total WAV files found:", len(all_wav_files))

# Create filename -> path mapping
wav_map = {
    os.path.basename(path): path
    for path in all_wav_files
}

# Check submission files
sample_submission["found"] = sample_submission["filename"].isin(wav_map)

print("Submission rows:", len(sample_submission))
print("Audio files found:", sample_submission["found"].sum())
print("Audio files missing:", (~sample_submission["found"]).sum())

print("\nMissing files:")
display(
    sample_submission.loc[~sample_submission["found"], ["filename"]].head(20)
)

In [ ]:
import pandas as pd
import os

base_path = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

# Load the two CSV files directly
test_df = pd.read_csv(os.path.join(base_path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(base_path, "sample_submission.csv"))

print("test.csv shape:", test_df.shape)
print("sample_submission.csv shape:", sample_submission.shape)

print("\nTest CSV columns:")
print(test_df.columns.tolist())

print("\nSubmission columns:")
print(sample_submission.columns.tolist())

print("\nFirst 5 test rows:")
print(test_df.head())

print("\nFirst 5 submission rows:")
print(sample_submission.head())

In [ ]:
# Compare filenames

test_names = set(test_df["filename"].astype(str))
submission_names = set(sample_submission["filename"].astype(str))

print("Test CSV filenames:", len(test_names))
print("Submission filenames:", len(submission_names))

print("\nTest files also in submission:", len(test_names & submission_names))
print("Test files NOT in submission:", len(test_names - submission_names))
print("Submission files NOT in test:", len(submission_names - test_names))

print("\n--- Test files NOT in submission ---")
print(sorted(test_names - submission_names)[:30])

print("\n--- Submission files NOT in test ---")
print(sorted(submission_names - test_names)[:30])

In [ ]:
# Verify actual WAV files against test.csv

import glob

test_audio_dir = os.path.join(base_path, "test")

actual_test_files = {
    os.path.basename(path)
    for path in glob.glob(os.path.join(test_audio_dir, "*.wav"))
}

test_csv_files = set(test_df["filename"].astype(str))

print("Actual WAV files in test folder:", len(actual_test_files))
print("Filenames in test.csv:", len(test_csv_files))

print("\nWAV files missing from test.csv:",
      len(actual_test_files - test_csv_files))

print("test.csv files missing from test folder:",
      len(test_csv_files - actual_test_files))

if actual_test_files - test_csv_files:
    print("\nExamples of WAV files not in test.csv:")
    print(sorted(actual_test_files - test_csv_files)[:20])

if test_csv_files - actual_test_files:
    print("\nExamples of test.csv files not in test folder:")
    print(sorted(test_csv_files - actual_test_files)[:20])

In [ ]:
import re

def get_audio_number(filename):
    match = re.search(r"audio_(\d+)\.wav", str(filename))
    return int(match.group(1)) if match else None

test_numbers = sorted(
    test_df["filename"].apply(get_audio_number).dropna().astype(int)
)

submission_numbers = sorted(
    sample_submission["filename"].apply(get_audio_number).dropna().astype(int)
)

print("Test filename number range:")
print(min(test_numbers), "to", max(test_numbers))

print("\nSubmission filename number range:")
print(min(submission_numbers), "to", max(submission_numbers))

print("\nFirst 30 test numbers:")
print(test_numbers[:30])

print("\nLast 30 test numbers:")
print(test_numbers[-30:])

print("\nFirst 30 submission numbers:")
print(submission_numbers[:30])

print("\nLast 30 submission numbers:")
print(submission_numbers[-30:])

In [ ]:
# Check whether submission filenames are simply a subset
# of some larger audio numbering scheme.

all_numbers = sorted(set(test_numbers) | set(submission_numbers))

print("Unique audio IDs across test + submission:", len(all_numbers))
print("Minimum ID:", min(all_numbers))
print("Maximum ID:", max(all_numbers))

# Show gaps in the overall numbering
missing_ids = [
    i for i in range(min(all_numbers), max(all_numbers) + 1)
    if i not in set(all_numbers)
]

print("\nNumber of missing IDs between min and max:", len(missing_ids))
print("First 50 missing IDs:")
print(missing_ids[:50])

In [ ]:
import os
import pandas as pd

base_path = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

print("Dataset_Final contents:\n")

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files)[:20]:
        print(f"{indent}  {file}")

    if len(files) > 20:
        print(f"{indent}  ... ({len(files)} files total)")

In [ ]:
# Inspect every CSV in Dataset_Final

import glob

csv_files = glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)

print("CSV files found:")
for path in csv_files:
    print("\n", path)

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3))
    except Exception as e:
        print("Could not read:", e)

In [ ]:
# Inspect every CSV in Dataset_Final

import glob

csv_files = glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)

print("CSV files found:")
for path in csv_files:
    print("\n", path)

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3))
    except Exception as e:
        print("Could not read:", e)

In [ ]:
train_df = pd.read_csv(os.path.join(base_path, "train.csv"))

train_numbers = sorted(
    train_df["filename"]
    .str.extract(r"audio_(\d+)\.wav")[0]
    .dropna()
    .astype(int)
)

test_numbers = sorted(
    test_df["filename"]
    .str.extract(r"audio_(\d+)\.wav")[0]
    .dropna()
    .astype(int)
)

submission_numbers = sorted(
    sample_submission["filename"]
    .str.extract(r"audio_(\d+)\.wav")[0]
    .dropna()
    .astype(int)
)

print("TRAIN")
print("count:", len(train_numbers))
print("range:", min(train_numbers), "to", max(train_numbers))

print("\nTEST")
print("count:", len(test_numbers))
print("range:", min(test_numbers), "to", max(test_numbers))

print("\nSUBMISSION")
print("count:", len(submission_numbers))
print("range:", min(submission_numbers), "to", max(submission_numbers))

In [ ]:
print("========== TRAIN ==========")
print(train_df.head(10))
print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\n========== TEST ==========")
print(test_df.head(10))
print("\nUnique test labels:")
print(test_df["label"].value_counts(dropna=False))

print("\n========== SAMPLE SUBMISSION ==========")
print(sample_submission.head(10))
print("\nUnique submission labels:")
print(sample_submission["label"].value_counts(dropna=False))

print("\n========== FILE COUNTS ==========")
print("Train CSV:", len(train_df))
print("Train WAV:", len(os.listdir(os.path.join(base_path, "train"))))
print("Test CSV:", len(test_df))
print("Test WAV:", len(os.listdir(os.path.join(base_path, "test"))))
print("Submission:", len(sample_submission))

In [ ]:
competition_path = "/kaggle/input/competitions/shl-hiring-assessment-2026"

print("Files/folders under competition directory:\n")

for item in sorted(os.listdir(competition_path)):
    full_path = os.path.join(competition_path, item)

    if os.path.isdir(full_path):
        print("[DIR ]", item)
    else:
        print("[FILE]", item)

In [ ]:
print("Number of test predictions:", len(test_predictions))
print("First 10 predictions:")
print(test_predictions[:10])

In [ ]:
import pandas as pd
import numpy as np
import os

# Create submission using the actual supplied test set
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": np.clip(test_predictions, 0, 5)
})

# Save submission
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission created successfully.")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

print("\nFirst 10 rows:")
print(submission.head(10))

print("\nPrediction statistics:")
print(submission["label"].describe())

print("\nMissing values:")
print(submission.isna().sum())

print("\nSaved to:")
print(submission_path)

In [ ]:
# Final submission validation

check = pd.read_csv("/kaggle/working/submission.csv")

print("Rows:", len(check))
print("Expected rows:", len(test_df))

print("Columns:", check.columns.tolist())

print("Missing filenames:", check["filename"].isna().sum())
print("Missing labels:", check["label"].isna().sum())

print("Labels below 0:", (check["label"] < 0).sum())
print("Labels above 5:", (check["label"] > 5).sum())

print("\nFilename match with test.csv:",
      check["filename"].tolist() == test_df["filename"].tolist())

print("\nFinal validation:")
if (
    len(check) == 216
    and check.columns.tolist() == ["filename", "label"]
    and check["filename"].notna().all()
    and check["label"].notna().all()
    and (check["label"] >= 0).all()
    and (check["label"] <= 5).all()
):
    print("✓ Submission file is valid for the supplied 216-sample test set.")
else:
    print("✗ Submission validation failed.")

In [ ]:
import os

print(os.path.exists("/kaggle/working/submission.csv"))
print(os.path.getsize("/kaggle/working/submission.csv"), "bytes")

# 40. Final Report

Add this as a Markdown cell after all your results.

# Final Report

## 1. Objective

The objective of this assessment is to predict a continuous grammar score between 0 and 5 for spoken English audio samples.

The available training data contains 769 labelled recordings and the test data contains 216 recordings.

## 2. Data Preparation

The WAV audio files were loaded using Librosa and converted to a consistent mono 16 kHz representation for feature extraction.

The recordings were checked for missing or unusually short/long files before model training.

## 3. Feature Engineering

Audio-based features were extracted from each recording.

The features include:

- Audio duration
- Speech duration
- Speech activity ratio
- RMS energy statistics
- Zero crossing rate
- Spectral centroid
- Spectral bandwidth
- Spectral rolloff
- MFCC statistics
- Pitch statistics

These features describe speaking characteristics such as energy, spectral behaviour, pitch variation and speech activity.

## 4. Modeling Approach

The problem was treated as a regression task because the required output is a continuous score between 0 and 5.

Random Forest and Extra Trees regression models were evaluated using five-fold cross-validation.

The final model was selected based on validation RMSE.

Predictions were clipped to the valid range of 0 to 5.

## 5. Validation

Five-fold cross-validation was used to evaluate the model.

### Final Training Validation Result

**Training OOF RMSE:** [VALUE FROM CELL 43]

**Training OOF Pearson Correlation:** [VALUE FROM CELL 43]

RMSE measures the magnitude of prediction errors, while Pearson Correlation measures how well the predicted scores follow the ranking and variation of the actual grammar scores.

## 6. Visualizations

The notebook includes:

- Grammar score distribution
- Audio duration distribution
- Feature correlation analysis
- Actual versus predicted scores
- Residual analysis
- Prediction error distribution
- Feature importance
- Test prediction distribution

## 7. Final Prediction

After validation, the selected model was retrained using all 769 labelled training samples.

The trained model was then used to generate predictions for the 216 test recordings.

The final predictions were constrained to the required 0–5 range and saved in the required submission format.